In [ ]:
import json

import matplotlib.pyplot as plt
from matplotlib.patches import Patch

In [ ]:
MASTER_LABEL = "Master"
LOAD_COLOR = "#7F8C8D"
MERGE_COLOR = "#5B7DB1"
ENCODE_COLOR = "#2E8B57"
OVERHEAD_COLOR = "#BFE3CC"
FAIL_COLORS = {
    "failed": "#D9534F",
    "worker_lost": "#F0AD4E",
}
FAIL_NAMES = {
    "failed": "Failed segment (retry)",
    "worker_lost": "Worker unreachable",
}

In [ ]:
Y_MARGIN_IN = 1.0
MAX_PIXELS = 60000
BAR_HEIGHT = 0.7

def load_log(path):
    with open(path, "r", encoding = "utf-8") as f:
        return json.load(f)

def bar_label(r, short = False):
    if short:
        return f"{r['segment']}" if r["attempt"] == 1 else f"{r['segment']}*"

    return f"S{r['segment']}" if r["attempt"] == 1 else f"S{r['segment']} ({r['attempt']})"

def text_width_in(text, fontsize):
    return len(text) * fontsize * 0.62 / 72 + fontsize * 0.6 / 72

def glyph_height_in(fontsize):
    return fontsize * 0.9 / 72

def label_mode(duration, text, scale, fontsize):
    bar_in = duration * scale

    if bar_in >= text_width_in(text, fontsize):
        return "h"

    if bar_in >= glyph_height_in(fontsize):
        return "v"

    return None

def plot_gantt(
    log,
    output = None,
    show = True,
    width = None,
    panels = None,
    max_panels = 6,
    label_fontsize = 7,
    fontsize = 9,
    short_labels = False,
    segment_labels = True,   # True: tulis label segmen (S1, S2, ...) di bar; False: tanpa label segmen
    title = True,
    encode_only = False,
    max_width = 250,
    dpi = 300,
):
    timeline = [r for r in log["timeline"] if r["end"] is not None]
    workers = log.get("workers") or sorted({r["worker_id"] for r in timeline})
    summary = log.get("summary", {})
    worker_summary = summary.get("workers", {})
    merge = None if encode_only else log.get("merge")

    load_time = 0.0 if encode_only else (log.get("load_video_time") or 0.0)
    shift = load_time

    has_merge = bool(merge and merge.get("duration"))
    rows = ([MASTER_LABEL] if (load_time > 0 or has_merge) else []) + list(workers)
    y_of = {name: index for index, name in enumerate(rows)}

    makespan = summary.get("makespan") or max((r["end"] for r in timeline), default = 0.0)
    encode_end = makespan + shift
    x_max = max(encode_end, (merge["end"] + shift) if has_merge else 0.0, 1e-6)

    items = [(bar_label(r, short_labels), r["start"] + shift, r["end"] + shift) for r in timeline] if segment_labels else []

    if load_time > 0:
        items.append(("Load", 0.0, load_time))

    if has_merge:
        items.append(("Merge", merge["start"] + shift, merge["end"] + shift))

    def count_unfit(scale):
        return sum(1 for t, a, b in items if label_mode(b - a, t, scale, label_fontsize) is None)

    if width is not None:
        axes_width = max(1.0, width - Y_MARGIN_IN)

        if panels is None:
            panels = next((p for p in range(1, max_panels + 1) if count_unfit(p * axes_width / x_max) == 0), max_panels)

        panels = max(1, int(panels))
        fig_width = width
    else:
        panels = 1
        max_axes = min(max_width, MAX_PIXELS / dpi) - Y_MARGIN_IN
        needed_h = max((text_width_in(t, label_fontsize) / (b - a) for t, a, b in items if b > a), default = 0)
        needed_v = max((glyph_height_in(label_fontsize) / (b - a) for t, a, b in items if b > a), default = 0)
        axes_width = x_max * needed_h

        if axes_width > max_axes:
            axes_width = min(max_axes, x_max * needed_v)

        axes_width = max(8.0, axes_width)
        fig_width = axes_width + Y_MARGIN_IN

    span = x_max / panels
    scale = axes_width / span

    vertical = [t for t, a, b in items if label_mode(b - a, t, scale, label_fontsize) == "v"]
    longest_vertical = max((text_width_in(t, label_fontsize) for t in vertical), default = 0)
    row_height = max(fontsize * 3.2 / 72, longest_vertical / BAR_HEIGHT + 0.04)  # min: muat label Y 2 baris

    panel_height = row_height * len(rows)
    xtick_height = fontsize * 2.2 / 72
    fig_height = panels * (panel_height + xtick_height) + (fontsize * 2.0 / 72 if title else 0) + 0.45

    fig, axes = plt.subplots(panels, 1, figsize = (fig_width, fig_height), squeeze = False)
    axes = axes[:, 0]
    edge = 0.3 if width is not None else 0.5
    skipped = []

    def home_panel(a, b):
        return max(range(panels), key = lambda k: min(b, (k + 1) * span) - max(a, k * span))

    def put_label(ax, k, a, b, text):
        lo, hi = max(a, k * span), min(b, (k + 1) * span)
        mode = label_mode(hi - lo, text, scale, label_fontsize)

        if mode is None:
            skipped.append(text)
            return

        ax.text(
            (lo + hi) / 2, ax_y[text_key(a, b)], text,
            ha = "center", va = "center",
            fontsize = label_fontsize,
            rotation = 90 if mode == "v" else 0,
            color = "white" if mode == "v" else "black",
            bbox = None if mode == "v" else dict(boxstyle = "round,pad=0.1", fc = "white", ec = "none", alpha = 0.75),
            clip_on = True,
        )

    ax_y = {}

    def text_key(a, b):
        return (round(a, 6), round(b, 6))

    has_overhead = False

    for k, ax in enumerate(axes):
        if load_time > 0:
            y = y_of[MASTER_LABEL]
            ax.barh(y, load_time, left = 0, height = BAR_HEIGHT, color = LOAD_COLOR, edgecolor = "black", linewidth = edge)

        if has_merge:
            y = y_of[MASTER_LABEL]
            merge_color = "#D9534F" if merge.get("error") else MERGE_COLOR
            ax.barh(y, merge["duration"], left = merge["start"] + shift, height = BAR_HEIGHT, color = merge_color, edgecolor = "black", linewidth = edge)

        for r in timeline:
            left = r["start"] + shift
            wall = r["end"] - r["start"]
            y = y_of[r["worker_id"]]
            encode = r.get("encoding_time")

            if r["status"] == "completed" and encode:
                encode = min(encode, wall)
                ax.barh(y, encode, left = left, height = BAR_HEIGHT, color = ENCODE_COLOR, edgecolor = "black", linewidth = edge)

                if wall - encode > 0:
                    has_overhead = True
                    ax.barh(y, wall - encode, left = left + encode, height = BAR_HEIGHT, color = OVERHEAD_COLOR, edgecolor = "black", hatch = "//", linewidth = edge)
            else:
                ax.barh(y, wall, left = left, height = BAR_HEIGHT, color = FAIL_COLORS.get(r["status"], "#999999"), edgecolor = "black", linewidth = edge)

        ax.axvline(encode_end, color = "black", linestyle = "--", linewidth = 0.8)

    for r in (timeline if segment_labels else []):
        a, b = r["start"] + shift, r["end"] + shift
        ax_y[text_key(a, b)] = y_of[r["worker_id"]]
        k = home_panel(a, b)
        put_label(axes[k], k, a, b, bar_label(r, short_labels))

    if load_time > 0:
        ax_y[text_key(0.0, load_time)] = y_of[MASTER_LABEL]
        k = home_panel(0.0, load_time)
        put_label(axes[k], k, 0.0, load_time, "Load")

    if has_merge:
        a, b = merge["start"] + shift, merge["end"] + shift
        ax_y[text_key(a, b)] = y_of[MASTER_LABEL]
        k = home_panel(a, b)
        put_label(axes[k], k, a, b, "Merge")

    ticklabels = []

    for name in rows:
        if name in worker_summary:
            ticklabels.append(f"{name}\nutil {worker_summary[name]['utilization'] * 100:.0f}%")
        else:
            ticklabels.append(name)

    for k, ax in enumerate(axes):
        ax.set_yticks(range(len(rows)))
        ax.set_yticklabels(ticklabels, fontsize = fontsize)
        ax.tick_params(axis = "x", labelsize = fontsize)
        ax.set_ylim(len(rows) - 0.5, -0.5)
        ax.set_xlim(k * span, (k + 1) * span + (x_max * 0.01 if k == panels - 1 else 0))
        ax.grid(axis = "x", linestyle = ":", alpha = 0.5)
        ax.set_axisbelow(True)
        ax.xaxis.set_major_locator(plt.MaxNLocator(nbins = max(5, int(axes_width / 0.7)), steps = [1, 2, 2.5, 5, 10]))

    axes[-1].set_xlabel("Time (s)", fontsize = fontsize)

    if title:
        text = f"Job {log.get('job_id')} | scheduler: {log.get('scheduler')} | makespan encode: {makespan:.2f}s"

        if load_time > 0:
            text += f" | load video: {load_time:.2f}s"

        if has_merge:
            text += f" | merge: {merge['duration']:.2f}s"

        axes[0].set_title(text, fontsize = fontsize, loc = "left")

    legend = []

    if load_time > 0:
        legend.append(Patch(facecolor = LOAD_COLOR, edgecolor = "black", label = "Load video"))

    if any(r["status"] == "completed" for r in timeline):
        legend.append(Patch(facecolor = ENCODE_COLOR, edgecolor = "black", label = "Encode"))

    if has_overhead:
        legend.append(Patch(facecolor = OVERHEAD_COLOR, edgecolor = "black", hatch = "//", label = "Overhead"))

    for status, color in FAIL_COLORS.items():
        if any(r["status"] == status for r in timeline):
            legend.append(Patch(facecolor = color, edgecolor = "black", label = FAIL_NAMES[status]))

    if has_merge:
        legend.append(Patch(facecolor = MERGE_COLOR, edgecolor = "black", label = "Merge"))

    if segment_labels and short_labels and any(r["attempt"] > 1 for r in timeline):
        legend.append(Patch(facecolor = "none", edgecolor = "none", label = "* = retry attempt"))

    fig.legend(handles = legend, loc = "lower center", ncol = len(legend), frameon = False, fontsize = fontsize,
               handlelength = 1.5, columnspacing = 1.0, bbox_to_anchor = (0.5, 0))
    fig.tight_layout(rect = (0, 0.35 / fig_height, 1, 1), h_pad = 0.6)

    if skipped:
        print(f"Peringatan: {len(skipped)} label tidak muat dan tidak ditulis (contoh: {skipped[:5]}). "
              f"Tambah panels, perbesar width, atau kecilkan label_fontsize.")

    if output:
        fig.savefig(output, dpi = dpi, bbox_inches = "tight", pad_inches = 0.02)
        print(f"Disimpan ke {output} ({fig_width:.2f} x {fig_height:.2f} inci, {panels} panel, {dpi} dpi)")

    if show:
        plt.show()
    else:
        plt.close(fig)

    return fig

In [ ]:
LOG_PATH = r"\\192.168.204.128\storages\Logs\job_623eb819.json"

In [ ]:
log = load_log(LOG_PATH)

# Paper IEEE 2 kolom, gambar membentang penuh kiri-kanan melintasi kedua kolom.
#   LaTeX : pakai \begin{figure*} ... \includegraphics[width=\textwidth]{...} ... \end{figure*}
#   Word  : sisipkan section break (continuous) dan set bagian gambar ke 1 kolom.
# Lebar area teks IEEE (kedua kolom + jarak antar kolom) = 7.16 inci (~18.2 cm).
# Jika template A4 milikmu sedikit berbeda, ukur di template: lebar kertas - margin kiri - margin kanan.
PAPER_TEXT_WIDTH_IN = 7.16

plot_gantt(
    log,
    output = f"Temp/gantt_{log['job_id']}_paper.png",
    show = False,
    width = PAPER_TEXT_WIDTH_IN,
    label_fontsize = 7,
    fontsize = 9,
    short_labels = True,
    segment_labels = True,   # False: tanpa label segmen di bar
    title = False,
    panels=1,
    encode_only = True,
);